In [0]:
%pip install nltk scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 11.1 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import nltk
from nltk.corpus import reuters
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import pandas as pd
import numpy as np

# Download required NLTK data
nltk.download('reuters')
nltk.download('punkt')


[nltk_data] Downloading package reuters to
[nltk_data]     /home/spark-a3867e84-da6c-46ce-b53e-51/nltk_data...
[nltk_data] Downloading package punkt to
[nltk_data]     /home/spark-a3867e84-da6c-46ce-b53e-51/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [0]:
# Create a sample dataset
sample_texts = [
    "Natural language processing is fascinating. It helps computers understand human language.",
    "Machine learning models can process and analyze text data efficiently.",
    "Text representation is crucial for NLP tasks and applications."
]

# Print basic statistics
print("Number of documents:", len(sample_texts))
print("Total characters:", sum(len(text) for text in sample_texts))


Number of documents: 3
Total characters: 221


In [0]:
# Create and configure CountVectorizer
bow_vectorizer = CountVectorizer(
    lowercase=True,
    token_pattern=r'\b\w+\b',
    min_df=1
)

# Generate BoW representation
X_bow = bow_vectorizer.fit_transform(sample_texts)

# Create a DataFrame for better visualization
bow_df = pd.DataFrame(
    X_bow.toarray(),
    columns=bow_vectorizer.get_feature_names_out()
)

print("\nBag-of-Words Representation:")
print(bow_df)



Bag-of-Words Representation:
   analyze  and  applications  can  ...  representation  tasks  text  understand
0        0    0             0    0  ...               0      0     0           1
1        1    1             0    1  ...               0      0     1           0
2        0    1             1    0  ...               1      1     1           0

[3 rows x 26 columns]


In [0]:
# Create and configure TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    token_pattern=r'\b\w+\b',
    min_df=1
)

# Generate TF-IDF representation
X_tfidf = tfidf_vectorizer.fit_transform(sample_texts)

# Create a DataFrame for better visualization
tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
)

print("\nTF-IDF Representation:")
print(tfidf_df)



TF-IDF Representation:
    analyze       and  applications  ...     tasks      text  understand
0  0.000000  0.000000      0.000000  ...  0.000000  0.000000     0.28196
1  0.330467  0.251329      0.000000  ...  0.000000  0.251329     0.00000
2  0.000000  0.273450      0.359554  ...  0.359554  0.273450     0.00000

[3 rows x 26 columns]


In [0]:
# Compare top terms by importance
def print_top_terms(vectorizer, matrix, n=5):
    sums = np.sum(matrix, axis=0)
    terms = vectorizer.get_feature_names_out()
    pairs = list(zip(terms, sums))
    pairs.sort(key=lambda x: x[1], reverse=True)
    
    print(f"\nTop {n} most important terms:")
    for term, score in pairs[:n]:
        print(f"{term}: {score:.3f}")

# Analyze both representations
print("\nBoW Analysis:")
print_top_terms(bow_vectorizer, X_bow.toarray())

print("\nTF-IDF Analysis:")
print_top_terms(tfidf_vectorizer, X_tfidf.toarray())



BoW Analysis:

Top 5 most important terms:
and: 2.000
is: 2.000
language: 2.000
text: 2.000
analyze: 1.000

TF-IDF Analysis:

Top 5 most important terms:
language: 0.564
and: 0.525
text: 0.525
is: 0.488
applications: 0.360


In [0]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate similarity matrices
bow_similarity = cosine_similarity(X_bow)
tfidf_similarity = cosine_similarity(X_tfidf)

print("\nDocument Similarity Matrix (BoW):")
print(pd.DataFrame(bow_similarity, columns=['Doc1', 'Doc2', 'Doc3']))

print("\nDocument Similarity Matrix (TF-IDF):")
print(pd.DataFrame(tfidf_similarity, columns=['Doc1', 'Doc2', 'Doc3']))



Document Similarity Matrix (BoW):
      Doc1      Doc2      Doc3
0  1.00000  0.000000  0.092450
1  0.00000  1.000000  0.210819
2  0.09245  0.210819  1.000000

Document Similarity Matrix (TF-IDF):
       Doc1      Doc2      Doc3
0  1.000000  0.000000  0.058638
1  0.000000  1.000000  0.137452
2  0.058638  0.137452  1.000000


Cosine similarity measures the cosine of the angle between two vectors.
In text analysis, it quantifies how similar two documents are based on their vector representations.
A value of 1 means the documents are identical in terms of their vectorized features, while 0 means they share no features in common.

Example: Compute cosine similarity between two sample vectors
```
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

vec1 = np.array([[1, 2, 3]])
vec2 = np.array([[1, 2, 3]])
vec3 = np.array([[0, 0, 1]])

similarity_1_2 = cosine_similarity(vec1, vec2)[0][0]
similarity_1_3 = cosine_similarity(vec1, vec3)[0][0]

print(f"Cosine similarity between vec1 and vec2: {similarity_1_2:.2f}")
print(f"Cosine similarity between vec1 and vec3: {similarity_1_3:.2f}")
```